# NeuSin

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.NeuSin)

class NeuSin(LinearReferenceClock):
    pass



In [3]:
model = pya.models.NeuSin()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "neusin"
model.metadata["data_type"] = "DNA methylation"  # Paper: The model is based on DNA methylation measurements.
model.metadata["species"] = "Homo sapiens"  # Paper: The study samples are Homo sapiens.
model.metadata["year"] = 2024
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Tong, Huige, et al. \"Cell-type specific epigenetic clocks to quantify biological age at cell-type resolution.\" Aging 16 (2024): 13452–13504."
model.metadata["doi"] = "https://doi.org/10.18632/aging.206184"
model.metadata["notes"] = "Neu-Sin is a neuron semi-intrinsic chronological-age clock: elastic-net regression was restricted to neuron age-DMCTs but fitted to methylation values not adjusted for brain cell fractions."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["brain cortex"]  # Paper: For training the clocks, we used one of the largest collections of prefrontal cortex (PFC) samples profiled with Illumina 450k technology, encompassing 416 samples with a wide age range (18–97 years).
model.metadata["predicts"] = ["chronological age"]  # Paper: Finally, elastic net predictors of chronological age are trained from the Neu-DMCTs parameterized by a penalty parameter.
model.metadata["training_target"] = ["chronological age"]  # Paper: Having identified the significant neuron-DMCTs, we next trained Elastic Net regression (glmnet R package) models (alpha = 0.5) for age, each parameterized by a different penalty parameter (lambda) value.
model.metadata["unit"] = ["years"]  # Paper: For training the clocks, we used one of the largest collections of prefrontal cortex (PFC) samples profiled with Illumina 450k technology, encompassing 416 samples with a wide age range (18–97 years).
model.metadata["model_type"] = "elastic net regression"  # Paper: Having identified the significant neuron-DMCTs, we next trained Elastic Net regression (glmnet R package) models (alpha = 0.5) for age, each parameterized by a different penalty parameter (lambda) value.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: For training the clocks, we used one of the largest collections of prefrontal cortex (PFC) samples profiled with Illumina 450k technology, encompassing 416 samples with a wide age range (18–97 years).
model.metadata["population"] = "adults"  # Paper: For training the clocks, we used one of the largest collections of prefrontal cortex (PFC) samples profiled with Illumina 450k technology, encompassing 416 samples with a wide age range (18–97 years).
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Andrew E. Teschendorff"
model.metadata["n_features"] = 672  # Paper: The official Neu-SinCoef.rda object contains 673 rows: one (Intercept) row and 672 non-intercept CpG coefficient rows.
model.metadata["citations"] = 25
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://raw.githubusercontent.com/Duzhaozhen/OmniAge/c10fbe8cb92957520fbff1d55ae1def0691252e5/OmniAgePy/src/omniage/data/CTS/Neu-Sin.csv"
supplementary_file_name = "coefficients.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
if str(df.columns[0]).startswith('Unnamed'):
    df = df.iloc[:, 1:]
mask = df['probe'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'coef'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['probe'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['coef'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Tong, Huige, et al. "Cell-type-specific and '
             'cell-type-independent DNA methylation clocks." Aging 16 (2024).',
 'clock_name': 'neusin',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.18632/aging.206184',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2024}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg10626816', 'cg13571388', 'cg17826530', 'cg06711298', 'cg08193650', 'cg10442729', 'cg17343483', 'cg20361600', 'cg21185289', 'cg16369288', 'cg22702772', 'cg26856080', 'cg06385118', 'cg18171715', 'cg18586891', 'cg05477834', 'cg15690342', 'cg18635552', 'cg18745317', 'cg25108022', 'cg00537387', 'cg08692175', 'cg25739875', 'cg18048071', 'cg19421584', 'cg

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[ -26.3359],
        [ 213.3675],
        [-104.1039],
        [-339.0749],
        [ -66.5274],
        [ 113.9317],
        [ 141.6260],
        [ -65.5180],
        [  -2.5413],
        [-400.5478]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
